In [1]:
import math
import random
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import psycopg
import requests

base_url = "http://localhost:8002"
endpoint = f"{base_url}/api/events/"

# pages with rough popularity weights
PAGES = [
    ("/", 30),
    ("/pricing", 20),
    ("/blog", 18),
    ("/docs", 15),
    ("/about", 10),
    ("/contact", 5),
    ("/careers", 2),
]
names = [p for p, _ in PAGES]
weights = [w for _, w in PAGES]

print(len(PAGES), "pages")

7 pages


In [2]:
# creds come from .env.compose so nothing is hardcoded here
env = dict(
    line.strip().split("=", 1)
    for line in Path("../.env.compose").read_text().splitlines()
    if "=" in line
)
DSN = (
    f"postgresql://{env['POSTGRES_USER']}:{env['POSTGRES_PASSWORD']}"
    f"@localhost:5432/{env['POSTGRES_DB']}"
)

with psycopg.connect(DSN) as conn:
    print("rows now:", conn.execute("select count(*) from eventmodel").fetchone()[0])

rows now: 2277


In [3]:
# optional - clear everything for a clean run
with psycopg.connect(DSN) as conn:
    conn.execute("TRUNCATE eventmodel")
    conn.commit()
print("wiped")

wiped


In [4]:
# traffic is busier midday, dead overnight
def diurnal_weight(hour: int) -> float:
    return 0.15 + 0.85 * max(0.0, math.sin((hour - 6) / 24 * 2 * math.pi))


for h in range(0, 24, 3):
    bar = "#" * int(diurnal_weight(h) * 40)
    print(f"{h:02d}:00  {bar}")

00:00  ######
03:00  ######
06:00  ######
09:00  ##############################
12:00  ########################################
15:00  ##############################
18:00  ######
21:00  ######


In [5]:
# backfill history - the API always stamps "now", so write straight to pg
DAYS_BACK = 7
EVENTS_PER_DAY = 400

rows = []
now = datetime.now(timezone.utc)

for day in range(DAYS_BACK):
    day_start = (now - timedelta(days=day)).replace(hour=0, minute=0, second=0, microsecond=0)
    # weekends are quieter
    day_scale = 0.5 if day_start.weekday() >= 5 else 1.0
    for _ in range(int(EVENTS_PER_DAY * day_scale)):
        hour = random.choices(range(24), weights=[diurnal_weight(h) for h in range(24)])[0]
        ts = day_start + timedelta(hours=hour, minutes=random.randint(0, 59), seconds=random.randint(0, 59))
        if ts > now:
            continue
        rows.append((ts, random.choices(names, weights=weights)[0], "", ts))

with psycopg.connect(DSN) as conn:
    conn.cursor().executemany(
        "INSERT INTO eventmodel (time, page, description, updated_at) VALUES (%s,%s,%s,%s)",
        rows,
    )
    conn.commit()

print(f"backfilled {len(rows)} events over {DAYS_BACK} days")

backfilled 2227 events over 7 days


In [6]:
# chunks are created per day of data - this is the hypertable partitioning
with psycopg.connect(DSN) as conn:
    n = conn.execute(
        "select count(*) from timescaledb_information.chunks where hypertable_name='eventmodel'"
    ).fetchone()[0]
    total = conn.execute("select count(*) from eventmodel").fetchone()[0]
print("chunks:", n, "| rows:", total)

chunks: 7 | rows: 2227


In [7]:
# burst of live traffic through the real API
BURST = 50

for _ in range(BURST):
    page = random.choices(names, weights=weights)[0]
    requests.post(endpoint, json={"page": page})

print("posted", BURST, "events via the API")

posted 50 events via the API


In [8]:
# continuous stream - fills up the 1 minute buckets
DURATION_SECONDS = 60

started = time.time()
sent = 0
while time.time() - started < DURATION_SECONDS:
    page = random.choices(names, weights=weights)[0]
    requests.post(endpoint, json={"page": page})
    sent += 1
    print(f"\r{sent:4d} events  ->  {page:<10}", end="")
    time.sleep(random.uniform(0.05, 0.6))

print(f"\ndone, sent {sent} events in {DURATION_SECONDS}s")

 182 events  ->  /         
done, sent 182 events in 60s
